# ARIX — Demostración del Backend
### Plataforma SaaS Marketplace Multi-Tienda

Este notebook consume la API  del backend de ARIX (FastAPI + MySQL) para demostrar el funcionamiento de cada módulo: autenticación, tiendas, productos, inventario, órdenes multi-tienda, facturación PDF, reseñas, favoritos, tickets de soporte, chat, notificaciones y auditoría.


backend : `uvicorn app.main:app --reload --port 8080`


In [5]:
import requests
import json
from datetime import datetime

BASE_URL = "http://127.0.0.1:8080/api"
PASSWORD = "Arix2026!"  # contraseña común

# Tokens y datos 
tokens = {}

ids = {}


def pretty(data):
    """Imprime una respuesta JSON de forma legible."""
    print(json.dumps(data, indent=2, ensure_ascii=False))

def call(method, path, token=None, **kwargs):
    """Realiza una llamada HTTP a la API y muestra resultado resumido."""
    headers = kwargs.pop("headers", {})
    if token:
        headers["Authorization"] = f"Bearer {token}"
    url = f"{BASE_URL}{path}"
    response = requests.request(method, url, headers=headers, **kwargs)
    print(f"{method} {path} -> {response.status_code}")
    try:
        data = response.json()
    except ValueError:
        data = {"raw": response.text}
    return response.status_code, data

In [6]:
# Verificación rápida de  backend 
status, data = call("GET", "/health")
pretty(data)

GET /health -> 200
{
  "success": true,
  "message": "ARIX API funcionando correctamente",
  "environment": "development"
}


## 1. Autenticación (módulo `users`)
 login de los tres roles del sistema usando los usuarios del seed, y el registro de un nuevo cliente.

In [7]:
#  Login Super Admin 
status, data = call("POST", "/auth/login", json={"email": "superadmin@arix.com", "password": PASSWORD})
tokens["super_admin"] = data["data"]["access_token"]
ids["super_admin_id"] = data["data"]["user"]["id"]
pretty(data["data"]["user"])

POST /auth/login -> 200
{
  "id": 1,
  "full_name": "Carlos Rivera",
  "email": "superadmin@arix.com",
  "phone": "+504 9999-0001",
  "address": "Tegucigalpa, Honduras",
  "profile_image_url": "/images/default-avatar.png",
  "status": "ACTIVE",
  "email_verified": true,
  "role": {
    "id": 3,
    "name": "ROLE_SUPER_ADMIN"
  },
  "created_at": "2026-06-17T23:35:51"
}


In [8]:
#  Login Admin de Tienda (TechStore HN) 
status, data = call("POST", "/auth/login", json={"email": "admin.techstore@arix.com", "password": PASSWORD})
tokens["admin_techstore"] = data["data"]["access_token"]
ids["admin_techstore_id"] = data["data"]["user"]["id"]
pretty(data["data"]["user"])

POST /auth/login -> 200
{
  "id": 2,
  "full_name": "María Fernández",
  "email": "admin.techstore@arix.com",
  "phone": "+504 9999-0002",
  "address": "Tegucigalpa, Honduras",
  "profile_image_url": "/images/default-avatar.png",
  "status": "ACTIVE",
  "email_verified": true,
  "role": {
    "id": 2,
    "name": "ROLE_STORE_ADMIN"
  },
  "created_at": "2026-06-17T23:35:51"
}


In [9]:
#  Login Admin de Tienda (ModaCasa) 
status, data = call("POST", "/auth/login", json={"email": "admin.modacasa@arix.com", "password": PASSWORD})
tokens["admin_modacasa"] = data["data"]["access_token"]
ids["admin_modacasa_id"] = data["data"]["user"]["id"]
pretty(data["data"]["user"])

POST /auth/login -> 200
{
  "id": 3,
  "full_name": "Jorge Martínez",
  "email": "admin.modacasa@arix.com",
  "phone": "+504 9999-0003",
  "address": "San Pedro Sula, Honduras",
  "profile_image_url": "/images/default-avatar.png",
  "status": "ACTIVE",
  "email_verified": true,
  "role": {
    "id": 2,
    "name": "ROLE_STORE_ADMIN"
  },
  "created_at": "2026-06-17T23:35:51"
}


In [52]:
#  Login Cliente (Ana López) 
status, data = call("POST", "/auth/login", json={"email": "ana.lopez@gmail.com", "password": PASSWORD})
tokens["ana"] = data["data"]["access_token"]
ids["ana_id"] = data["data"]["user"]["id"]
pretty(data["data"]["user"])

POST /auth/login -> 200
{
  "id": 4,
  "full_name": "Ana López",
  "email": "ana.lopez@gmail.com",
  "phone": "+504 9999-1001",
  "address": "Col. Palmira, Tegucigalpa",
  "profile_image_url": "/images/default-avatar.png",
  "status": "ACTIVE",
  "email_verified": true,
  "role": {
    "id": 1,
    "name": "ROLE_CLIENT"
  },
  "created_at": "2026-06-17T23:35:51"
}


In [11]:
#  Login Cliente (Luis Gómez) 
status, data = call("POST", "/auth/login", json={"email": "luis.gomez@gmail.com", "password": PASSWORD})
tokens["luis"] = data["data"]["access_token"]
ids["luis_id"] = data["data"]["user"]["id"]
pretty(data["data"]["user"])

POST /auth/login -> 200
{
  "id": 5,
  "full_name": "Luis Gómez",
  "email": "luis.gomez@gmail.com",
  "phone": "+504 9999-1002",
  "address": "Barrio Río de Piedras, SPS",
  "profile_image_url": "/images/default-avatar.png",
  "status": "ACTIVE",
  "email_verified": true,
  "role": {
    "id": 1,
    "name": "ROLE_CLIENT"
  },
  "created_at": "2026-06-17T23:35:51"
}


In [12]:
#  Registro de un nuevo cliente  SOLO clientes pueden autorregistrarse
new_client_email = f"cliente.demo.{int(datetime.now().timestamp())}@gmail.com"
status, data = call("POST", "/auth/register", json={
    "full_name": "Cliente Demo",
    "email": new_client_email,
    "password": PASSWORD,
    "phone": "+504 9999-2000",
    "address": "Col. Las Uvas, Tegucigalpa"
})
tokens["demo_client"] = data["data"]["access_token"]
ids["demo_client_id"] = data["data"]["user"]["id"]
pretty(data["data"]["user"])

POST /auth/register -> 200
{
  "id": 9,
  "full_name": "Cliente Demo",
  "email": "cliente.demo.1781823855@gmail.com",
  "phone": "+504 9999-2000",
  "address": "Col. Las Uvas, Tegucigalpa",
  "profile_image_url": "/images/default-avatar.png",
  "status": "ACTIVE",
  "email_verified": false,
  "role": {
    "id": 1,
    "name": "ROLE_CLIENT"
  },
  "created_at": "2026-06-18T17:04:15"
}


In [13]:
#  Mi perfil (cualquier usuario autenticado) 
status, data = call("GET", "/users/me", token=tokens["ana"])
pretty(data)

GET /users/me -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "id": 4,
    "full_name": "Ana López",
    "email": "ana.lopez@gmail.com",
    "phone": "+504 9999-1001",
    "address": "Col. Palmira, Tegucigalpa",
    "profile_image_url": "/images/default-avatar.png",
    "status": "ACTIVE",
    "email_verified": true,
    "role": {
      "id": 1,
      "name": "ROLE_CLIENT"
    },
    "created_at": "2026-06-17T23:35:51"
  },
  "timestamp": "2026-06-18T23:04:15.286705Z"
}


## 2. Tiendas 
Listado público del marketplace y consulta del perfil de tienda como Admin.

In [14]:
#  Listado público de tiendas activas 
status, data = call("GET", "/stores")
pretty(data)

GET /stores -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "content": [
      {
        "id": 1,
        "business_name": "TechStore HN",
        "slug": "techstore-hn",
        "logo_url": "/images/stores/techstore-logo.png",
        "banner_url": "/images/stores/techstore-banner.png",
        "status": "ACTIVE"
      },
      {
        "id": 2,
        "business_name": "ModaCasa",
        "slug": "modacasa",
        "logo_url": "/images/stores/modacasa-logo.png",
        "banner_url": "/images/stores/modacasa-banner.png",
        "status": "ACTIVE"
      }
    ],
    "page_number": 1,
    "page_size": 10,
    "total_elements": 2,
    "total_pages": 1,
    "last": true
  },
  "timestamp": "2026-06-18T23:04:15.309399Z"
}


In [15]:
#  Mi tienda (Admin de TechStore) 
status, data = call("GET", "/store/me", token=tokens["admin_techstore"])
ids["techstore_id"] = data["data"]["id"]
pretty(data)

GET /store/me -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "id": 1,
    "business_name": "TechStore HN",
    "slug": "techstore-hn",
    "description": "Tecnología, gadgets y accesorios electrónicos de alta calidad.",
    "logo_url": "/images/stores/techstore-logo.png",
    "banner_url": "/images/stores/techstore-banner.png",
    "contact_email": "contacto@techstore.com",
    "contact_phone": "+504 2222-1111",
    "contact_address": "Tegucigalpa, Honduras",
    "status": "ACTIVE",
    "admin": {
      "id": 2,
      "full_name": "María Fernández",
      "email": "admin.techstore@arix.com"
    },
    "profile": {
      "id": 1,
      "tagline": "La mejor tecnología al alcance de tu mano",
      "about": "TechStore HN es una tienda especializada en tecnología, con más de 5 años de experiencia ofreciendo productos originales y garantía oficial.",
      "social_facebook": "facebook.com/techstorehn",
      "social_instagram": "instagram.com/techstorehn",
      "social_webs

In [16]:
#  Mi tienda (Admin de ModaCasa) 
status, data = call("GET", "/store/me", token=tokens["admin_modacasa"])
ids["modacasa_id"] = data["data"]["id"]
pretty(data)

GET /store/me -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "id": 2,
    "business_name": "ModaCasa",
    "slug": "modacasa",
    "description": "Moda y artículos para el hogar con estilo moderno.",
    "logo_url": "/images/stores/modacasa-logo.png",
    "banner_url": "/images/stores/modacasa-banner.png",
    "contact_email": "contacto@modacasa.com",
    "contact_phone": "+504 2222-2222",
    "contact_address": "San Pedro Sula, Honduras",
    "status": "ACTIVE",
    "admin": {
      "id": 3,
      "full_name": "Jorge Martínez",
      "email": "admin.modacasa@arix.com"
    },
    "profile": {
      "id": 2,
      "tagline": "Estilo y comodidad para tu hogar",
      "about": "ModaCasa ofrece ropa, decoración y artículos para el hogar combinando tendencias actuales con precios accesibles.",
      "social_facebook": "facebook.com/modacasa",
      "social_instagram": "instagram.com/modacasa",
      "social_website": "www.modacasa.com",
      "tax_id": "RTN-0501-8765-4321"
 

## 3. Categorías y Productos (módulos `categories`, `products`, `inventory`)
  catálogo global, filtros de búsqueda y el inventario de cada tienda.

In [17]:
#  Árbol de categorías 
status, data = call("GET", "/categories")
pretty(data)

GET /categories -> 200
{
  "success": true,
  "message": "OK",
  "data": [
    {
      "id": 1,
      "name": "Electrónica",
      "slug": "electronica",
      "description": "Dispositivos electrónicos y gadgets",
      "parent_id": null,
      "children": [
        {
          "id": 4,
          "name": "Smartphones",
          "slug": "smartphones",
          "description": "Teléfonos inteligentes",
          "parent_id": 1,
          "children": []
        },
        {
          "id": 5,
          "name": "Audio",
          "slug": "audio",
          "description": "Audífonos, parlantes y equipos de sonido",
          "parent_id": 1,
          "children": []
        }
      ]
    },
    {
      "id": 2,
      "name": "Hogar",
      "slug": "hogar",
      "description": "Artículos y decoración para el hogar",
      "parent_id": null,
      "children": [
        {
          "id": 6,
          "name": "Decoración",
          "slug": "decoracion",
          "description": "Artículos dec

In [18]:
#  Catálogo completo (todos los productos activos) 
status, data = call("GET", "/products", params={"size": 20})
for p in data["data"]["content"]:
    print(f"#{p['id']:>2} | {p['name']:<40} | L. {p['price']:>10} | {p['store']['business_name']}")

GET /products -> 200
# 1 | Smartphone ARIX Pro 256GB                | L.   12999.00 | TechStore HN
# 2 | Audífonos Inalámbricos Premium           | L.    2499.00 | TechStore HN
# 3 | Smartwatch Fitness Tracker               | L.    1899.00 | TechStore HN
# 4 | Parlante Bluetooth Portátil              | L.    1299.00 | TechStore HN
# 5 | Set de Cojines Decorativos (4 piezas)    | L.     749.00 | ModaCasa
# 6 | Camisa Casual de Lino Hombre             | L.     549.00 | ModaCasa
# 7 | Vestido Casual Verano Mujer              | L.     699.00 | ModaCasa
# 8 | Lámpara de Mesa Minimalista              | L.     459.00 | ModaCasa


In [19]:
#  Filtro: productos de Electrónica ordenados por precio ascendente 
status, data = call("GET", "/products", params={"category_id": 1, "sort_by": "price", "sort_dir": "asc"})
for p in data["data"]["content"]:
    print(f"#{p['id']:>2} | {p['name']:<40} | L. {p['price']}")

GET /products -> 200
# 3 | Smartwatch Fitness Tracker               | L. 1899.00


In [53]:
#  Detalle de un producto específico 
status, data = call("GET", "/products/2")
pretty(data["data"])

GET /products/2 -> 200
{
  "id": 2,
  "name": "Audífonos Inalámbricos Premium",
  "slug": "audifonos-inalambricos-premium",
  "description": "Audífonos con cancelación activa de ruido, hasta 30 horas de batería y sonido de alta fidelidad.",
  "price": "2499.00",
  "sku": "TS-AU-002",
  "status": "ACTIVE",
  "is_moderated": true,
  "sales_count": 0,
  "rating_avg": "0.00",
  "rating_count": 0,
  "store": {
    "id": 1,
    "business_name": "TechStore HN",
    "slug": "techstore-hn"
  },
  "category": {
    "id": 5,
    "name": "Audio",
    "slug": "audio"
  },
  "images": [
    {
      "id": 3,
      "image_url": "/images/products/audifonos-premium-1.png",
      "is_primary": true,
      "display_order": 1
    },
    {
      "id": 4,
      "image_url": "/images/products/audifonos-premium-2.png",
      "is_primary": false,
      "display_order": 2
    }
  ],
  "inventory": {
    "product_id": 2,
    "stock_quantity": 8,
    "min_stock": 10,
    "is_low_stock": true,
    "is_out_of_stock"

In [21]:
#  Inventario de mis productos (Admin TechStore) 
status, data = call("GET", "/store/products/1/inventory", token=tokens["admin_techstore"])
pretty(data)

GET /store/products/1/inventory -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "product_id": 1,
    "stock_quantity": 25,
    "min_stock": 5,
    "is_low_stock": false,
    "is_out_of_stock": false
  },
  "timestamp": "2026-06-18T23:04:15.486487Z"
}


In [22]:
#  Productos con stock bajo (Admin TechStore) 
status, data = call("GET", "/store/inventory/low-stock", token=tokens["admin_techstore"])
pretty(data)

GET /store/inventory/low-stock -> 200
{
  "success": true,
  "message": "OK",
  "data": [
    {
      "product_id": 2,
      "stock_quantity": 8,
      "min_stock": 10,
      "is_low_stock": true,
      "is_out_of_stock": false
    },
    {
      "product_id": 3,
      "stock_quantity": 0,
      "min_stock": 5,
      "is_low_stock": true,
      "is_out_of_stock": true
    }
  ],
  "timestamp": "2026-06-18T23:04:15.507718Z"
}


In [23]:
#  Crear un nuevo producto (Admin ModaCasa) 
status, data = call("POST", "/store/products", token=tokens["admin_modacasa"], json={
    "category_id": 7,
    "name": "Bufanda Tejida Invierno",
    "slug": "bufanda-tejida-invierno",
    "description": "Bufanda de lana tejida a mano, ideal para climas fríos.",
    "price": 299.00,
    "sku": "MC-BU-005",
    "initial_stock": 15,
    "min_stock": 3
})
ids["new_product_id"] = data["data"]["id"]
pretty(data["data"])

POST /store/products -> 200
{
  "id": 9,
  "name": "Bufanda Tejida Invierno",
  "slug": "bufanda-tejida-invierno",
  "description": "Bufanda de lana tejida a mano, ideal para climas fríos.",
  "price": "299.00",
  "sku": "MC-BU-005",
  "status": "ACTIVE",
  "is_moderated": true,
  "sales_count": 0,
  "rating_avg": "0.00",
  "rating_count": 0,
  "store": {
    "id": 2,
    "business_name": "ModaCasa",
    "slug": "modacasa"
  },
  "category": {
    "id": 7,
    "name": "Ropa para Hombre",
    "slug": "ropa-hombre"
  },
  "images": [],
  "inventory": {
    "product_id": 9,
    "stock_quantity": 15,
    "min_stock": 3,
    "is_low_stock": false,
    "is_out_of_stock": false
  },
  "created_at": "2026-06-18T17:04:15"
}


## 4. Órdenes multi-tienda (módulo `orders`)
CASO:
Ana compra productos de dos tiendas distintas en un solo checkout. El backend debe generar automáticamente una orden principal y una suborden por cada tienda.

In [24]:
#  Checkout multi-tienda: 1 producto de TechStore + 1 de ModaCasa 
status, data = call("POST", "/orders/checkout", token=tokens["ana"], json={
    "items": [
        {"product_id": 1, "quantity": 1},
        {"product_id": 5, "quantity": 1}
    ],
    "shipping_address": "Col. Palmira, Tegucigalpa, Honduras",
    "payment_method": "CREDIT_CARD",
    "card_last_digits": "4321"
})
ids["order_id"] = data["data"]["id"]
pretty(data["data"])

POST /orders/checkout -> 200
{
  "id": 1,
  "order_number": "ORD-1001",
  "total_amount": "15810.20",
  "status": "CONFIRMED",
  "shipping_address": "Col. Palmira, Tegucigalpa, Honduras",
  "payment_method": "CREDIT_CARD",
  "store_orders": [
    {
      "id": 1,
      "sub_order_number": "ORD-1001-1",
      "store": {
        "id": 1,
        "business_name": "TechStore HN",
        "slug": "techstore-hn",
        "logo_url": "/images/stores/techstore-logo.png"
      },
      "subtotal": "12999.00",
      "tax_amount": "1949.85",
      "total": "14948.85",
      "status": "CONFIRMED",
      "items": [
        {
          "id": 1,
          "product_id": 1,
          "product_name": "Smartphone ARIX Pro 256GB",
          "unit_price": "12999.00",
          "quantity": 1,
          "line_total": "12999.00"
        }
      ],
      "created_at": "2026-06-18T17:04:15"
    },
    {
      "id": 2,
      "sub_order_number": "ORD-1001-2",
      "store": {
        "id": 2,
        "business_na

In [25]:
#  Verificamos que se generaron 2 SUBÓRDENES, una por cada tienda 
order = data["data"]
print(f"Orden principal: {order['order_number']}")
print(f"Total general: L. {order['total_amount']}\n")
for so in order["store_orders"]:
    print(f"  Suborden {so['sub_order_number']} | Tienda: {so['store']['business_name']} | Total: L. {so['total']} | Estado: {so['status']}")

Orden principal: ORD-1001
Total general: L. 15810.20

  Suborden ORD-1001-1 | Tienda: TechStore HN | Total: L. 14948.85 | Estado: CONFIRMED
  Suborden ORD-1001-2 | Tienda: ModaCasa | Total: L. 861.35 | Estado: CONFIRMED


In [26]:
#  Historial de órdenes de Ana 
status, data = call("GET", "/orders", token=tokens["ana"])
pretty(data)

GET /orders -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "content": [
      {
        "id": 1,
        "order_number": "ORD-1001",
        "total_amount": "15810.20",
        "status": "CONFIRMED",
        "created_at": "2026-06-18T17:04:15"
      }
    ],
    "page_number": 1,
    "page_size": 10,
    "total_elements": 1,
    "total_pages": 1,
    "last": true
  },
  "timestamp": "2026-06-18T23:04:15.690460Z"
}


In [27]:
#  El admin de TechStore ve su pedido recibido 
status, data = call("GET", "/store/orders", token=tokens["admin_techstore"])
pretty(data)

GET /store/orders -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "content": [
      {
        "id": 1,
        "sub_order_number": "ORD-1001-1",
        "order_number": "ORD-1001",
        "customer_name": "Ana López",
        "subtotal": "12999.00",
        "tax_amount": "1949.85",
        "total": "14948.85",
        "status": "CONFIRMED",
        "items": [
          {
            "id": 1,
            "product_id": 1,
            "product_name": "Smartphone ARIX Pro 256GB",
            "unit_price": "12999.00",
            "quantity": 1,
            "line_total": "12999.00"
          }
        ],
        "created_at": "2026-06-18T17:04:15"
      }
    ],
    "page_number": 1,
    "page_size": 10,
    "total_elements": 1,
    "total_pages": 1,
    "last": true
  },
  "timestamp": "2026-06-18T23:04:15.719463Z"
}


In [28]:
#  El admin de TechStore actualiza el estado del pedido: CONFIRMED -> PREPARING 
store_order_id = data["data"]["content"][0]["id"]
status, data = call("PUT", f"/store/orders/{store_order_id}/status", token=tokens["admin_techstore"], json={"status": "PREPARING"})
pretty(data)

PUT /store/orders/1/status -> 200
{
  "success": true,
  "message": "Estado del pedido actualizado correctamente",
  "data": {
    "id": 1,
    "sub_order_number": "ORD-1001-1",
    "store": {
      "id": 1,
      "business_name": "TechStore HN",
      "slug": "techstore-hn",
      "logo_url": "/images/stores/techstore-logo.png"
    },
    "subtotal": "12999.00",
    "tax_amount": "1949.85",
    "total": "14948.85",
    "status": "PREPARING",
    "items": [
      {
        "id": 1,
        "product_id": 1,
        "product_name": "Smartphone ARIX Pro 256GB",
        "unit_price": "12999.00",
        "quantity": 1,
        "line_total": "12999.00"
      }
    ],
    "created_at": "2026-06-18T17:04:15"
  },
  "timestamp": "2026-06-18T23:04:15.777258Z"
}


In [29]:
#  Avanzamos el pedido: PREPARING -> SHIPPED -> DELIVERED 
for next_status in ["SHIPPED", "DELIVERED"]:
    status, data = call("PUT", f"/store/orders/{store_order_id}/status", token=tokens["admin_techstore"], json={"status": next_status})
    print(f"  Nuevo estado: {data['data']['status']}")

PUT /store/orders/1/status -> 200
  Nuevo estado: SHIPPED
PUT /store/orders/1/status -> 200
  Nuevo estado: DELIVERED


In [30]:
#  Segunda compra: Luis compra solo de ModaCasa (orden de una sola tienda) 
status, data = call("POST", "/orders/checkout", token=tokens["luis"], json={
    "items": [{"product_id": 6, "quantity": 2}],
    "shipping_address": "Barrio Río de Piedras, San Pedro Sula, Honduras",
    "payment_method": "DEBIT_CARD",
    "card_last_digits": "9876"
})
ids["order_2_id"] = data["data"]["id"]
pretty(data["data"])

POST /orders/checkout -> 200
{
  "id": 2,
  "order_number": "ORD-1002",
  "total_amount": "1262.70",
  "status": "CONFIRMED",
  "shipping_address": "Barrio Río de Piedras, San Pedro Sula, Honduras",
  "payment_method": "DEBIT_CARD",
  "store_orders": [
    {
      "id": 3,
      "sub_order_number": "ORD-1002-1",
      "store": {
        "id": 2,
        "business_name": "ModaCasa",
        "slug": "modacasa",
        "logo_url": "/images/stores/modacasa-logo.png"
      },
      "subtotal": "1098.00",
      "tax_amount": "164.70",
      "total": "1262.70",
      "status": "CONFIRMED",
      "items": [
        {
          "id": 3,
          "product_id": 6,
          "product_name": "Camisa Casual de Lino Hombre",
          "unit_price": "549.00",
          "quantity": 2,
          "line_total": "1098.00"
        }
      ],
      "created_at": "2026-06-18T17:04:15"
    }
  ],
  "payment": {
    "id": 2,
    "payment_method": "DEBIT_CARD",
    "card_last_digits": "9876",
    "amount": "12

## 5. Facturación PDF (módulo `invoices`)
Generar las facturas de la orden multi-tienda de 
Ana: una por cada tienda y una consolidada, y descargamos el PDF de una de ellas.

In [31]:
#  Generar/listar facturas de la orden multi-tienda 
status, data = call("GET", f"/orders/{ids['order_id']}/invoices", token=tokens["ana"])
for inv in data["data"]:
    print(f"{inv['invoice_number']:<25} | Tipo: {inv['type']:<13} | Total: L. {inv['total']}")
ids["invoice_id"] = data["data"][0]["id"]

GET /orders/1/invoices -> 200
INV-1001-1                | Tipo: STORE         | Total: L. 14948.85
INV-1001-2                | Tipo: STORE         | Total: L. 861.35
INV-1001-CONSOLIDATED     | Tipo: CONSOLIDATED  | Total: L. 15810.20


In [32]:
#  Descargar el PDF de la primera factura y guardarlo localmente 
response = requests.get(
    f"{BASE_URL}/invoices/{ids['invoice_id']}/download",
    headers={"Authorization": f"Bearer {tokens['ana']}"}
)
filename = "factura_demo.pdf"
with open(filename, "wb") as f:
    f.write(response.content)
print(f"PDF descargado: {filename} ({len(response.content)} bytes)")

PDF descargado: factura_demo.pdf (55994 bytes)


## 6. Reseñas y Favoritos 
CASO
Ana reseña el producto que compró (solo puede reseñar productos comprados), y agrega productos a favoritos.

In [33]:
#  Ana reseña el Smartphone que compró 
status, data = call("POST", "/products/1/reviews", token=tokens["ana"], json={
    "rating": 5,
    "comment": "Excelente teléfono, la cámara es espectacular y la batería dura todo el día."
})
pretty(data["data"])

POST /products/1/reviews -> 200
{
  "id": 1,
  "product_id": 1,
  "rating": 5,
  "comment": "Excelente teléfono, la cámara es espectacular y la batería dura todo el día.",
  "status": "VISIBLE",
  "customer": {
    "id": 4,
    "full_name": "Ana López",
    "profile_image_url": "/images/default-avatar.png"
  },
  "created_at": "2026-06-18T17:04:16"
}


In [34]:
#  Intento fallido: reseñar un producto que NO ha comprado  
status, data = call("POST", "/products/8/reviews", token=tokens["ana"], json={"rating": 4})
pretty(data)

POST /products/8/reviews -> 403
{
  "success": false,
  "message": "Solo puedes reseñar productos que hayas comprado",
  "data": null,
  "timestamp": "2026-06-18T23:04:16.276559Z"
}


In [35]:
#  Verificamos que el rating del producto se actualizó automáticamente 
status, data = call("GET", "/products/1")
print(f"Rating promedio: {data['data']['rating_avg']} ({data['data']['rating_count']} reseñas)")

GET /products/1 -> 200
Rating promedio: 5.00 (1 reseñas)


In [36]:
#  Ana agrega productos a favoritos 
for product_id in [3, 8]:
    status, data = call("POST", "/favorites", token=tokens["ana"], json={"product_id": product_id})

status, data = call("GET", "/favorites", token=tokens["ana"])
pretty(data)

POST /favorites -> 200
POST /favorites -> 200
GET /favorites -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "content": [
      {
        "id": 1,
        "product": {
          "id": 3,
          "name": "Smartwatch Fitness Tracker",
          "slug": "smartwatch-fitness-tracker",
          "price": "1899.00",
          "status": "ACTIVE",
          "primary_image_url": "/images/products/smartwatch-fitness-1.png",
          "rating_avg": "0.00",
          "rating_count": 0,
          "sales_count": 0,
          "store": {
            "id": 1,
            "business_name": "TechStore HN",
            "slug": "techstore-hn"
          },
          "category": {
            "id": 1,
            "name": "Electrónica",
            "slug": "electronica"
          }
        },
        "created_at": "2026-06-18T17:04:16"
      },
      {
        "id": 2,
        "product": {
          "id": 8,
          "name": "Lámpara de Mesa Minimalista",
          "slug": "lampara-mesa-minima

## 7. Tickets de soporte 
CASO
Luis crea un ticket dirigido a TechStore, el admin responde, y se verifica el cambio automático de estado.

In [37]:
#  Luis crea un ticket dirigido a TechStore HN 
status, data = call("POST", "/tickets", token=tokens["luis"], json={
    "store_id": ids["techstore_id"],
    "subject": "Consulta sobre garantía del smartphone",
    "description": "Quisiera saber el proceso para reclamar garantía si el dispositivo falla.",
    "priority": "MEDIUM"
})
ids["ticket_id"] = data["data"]["id"]
print(f"Ticket creado: {data['data']['ticket_number']} | Estado: {data['data']['status']}")

POST /tickets -> 200
Ticket creado: TCK-0001 | Estado: OPEN


In [38]:
#  El admin de TechStore responde (esto cambia el estado a IN_PROGRESS automáticamente) 
status, data = call("POST", f"/tickets/{ids['ticket_id']}/messages", token=tokens["admin_techstore"], json={
    "message": "Hola Luis, la garantía cubre 12 meses por defectos de fábrica. Puedes acercarte a la tienda o coordinamos un envío."
})
print(f"Estado del ticket tras la respuesta: {data['data']['status']}")
for m in data["data"]["messages"]:
    print(f"  [{m['sender']['full_name']}]: {m['message']}")

POST /tickets/1/messages -> 200
Estado del ticket tras la respuesta: IN_PROGRESS
  [Luis Gómez]: Quisiera saber el proceso para reclamar garantía si el dispositivo falla.
  [María Fernández]: Hola Luis, la garantía cubre 12 meses por defectos de fábrica. Puedes acercarte a la tienda o coordinamos un envío.


In [39]:
#  El admin marca el ticket como resuelto 
status, data = call("PUT", f"/tickets/{ids['ticket_id']}/status", token=tokens["admin_techstore"], json={"status": "RESOLVED"})
print(f"Estado final: {data['data']['status']}")

PUT /tickets/1/status -> 200
Estado final: RESOLVED


## 8. Chat 
Probamos la parte HTTP del chat ( mensajes). La parte del push en tiempo real, requiere un cliente  se documenta al final de esta sección.

In [40]:
#  Ana inicia una conversación con el admin de TechStore 
status, data = call("POST", "/chats", token=tokens["ana"], json={
    "other_user_id": ids["admin_techstore_id"],
    "store_id": ids["techstore_id"]
})
ids["chat_id"] = data["data"]["id"]
pretty(data["data"])

POST /chats -> 200
{
  "id": 1,
  "other_user": {
    "id": 2,
    "full_name": "María Fernández",
    "profile_image_url": "/images/default-avatar.png"
  },
  "store": {
    "id": 1,
    "business_name": "TechStore HN",
    "slug": "techstore-hn"
  },
  "messages": [],
  "created_at": "2026-06-18T17:04:16",
  "updated_at": "2026-06-18T17:04:16"
}


In [41]:
#  Ana envía un mensaje 
status, data = call("POST", f"/chats/{ids['chat_id']}/messages", token=tokens["ana"], json={
    "content": "¡Hola! ¿Tienen disponible el Smartphone ARIX Pro en color negro?"
})
pretty(data["data"]["messages"])

POST /chats/1/messages -> 200
[
  {
    "id": 1,
    "sender": {
      "id": 4,
      "full_name": "Ana López",
      "profile_image_url": "/images/default-avatar.png"
    },
    "content": "¡Hola! ¿Tienen disponible el Smartphone ARIX Pro en color negro?",
    "is_read": false,
    "created_at": "2026-06-18T17:04:16"
  }
]


In [42]:
#  El admin responde 
status, data = call("POST", f"/chats/{ids['chat_id']}/messages", token=tokens["admin_techstore"], json={
    "content": "Hola Ana, sí tenemos disponibilidad en color negro y plateado."
})
for m in data["data"]["messages"]:
    print(f"  [{m['sender']['full_name']}]: {m['content']}")

POST /chats/1/messages -> 200
  [Ana López]: ¡Hola! ¿Tienen disponible el Smartphone ARIX Pro en color negro?
  [María Fernández]: Hola Ana, sí tenemos disponibilidad en color negro y plateado.


In [43]:
#  Bandeja de conversaciones de Ana (con conteo de no leídos) 
status, data = call("GET", "/chats", token=tokens["ana"])
pretty(data)

GET /chats -> 200
{
  "success": true,
  "message": "OK",
  "data": [
    {
      "id": 1,
      "other_user": {
        "id": 2,
        "full_name": "María Fernández",
        "profile_image_url": "/images/default-avatar.png"
      },
      "store": {
        "id": 1,
        "business_name": "TechStore HN",
        "slug": "techstore-hn"
      },
      "last_message": "Hola Ana, sí tenemos disponibilidad en color negro y plateado.",
      "unread_count": 1,
      "updated_at": "2026-06-18T17:04:16"
    }
  ],
  "timestamp": "2026-06-18T23:04:16.735694Z"
}


**Sobre el WebSocket en tiempo real:** la conexión se abre así desde el frontend (o cualquier cliente WebSocket):

```
ws://127.0.0.1:8080/api/ws/chat?token=<access_token>
```

Mientras esa conexión esté abierta, cualquier mensaje enviado por HTTP (`POST /api/chats/{id}/messages`) hacia ese usuario se empuja instantáneamente por el socket. Esto no se puede demostrar directamente desde `requests` en este notebook porque requiere un cliente WebSocket asíncrono; se documenta aquí como evidencia de la arquitectura.

## 9. Notificaciones y Auditoría 
verificar que las acciones anteriores (checkout, cambios de estado, tickets) generaron notificaciones automáticas, y que el Super Admin puede consultar el historial de auditoría.

In [44]:
#  Notificaciones de Ana (debe incluir 'Pedido confirmado', respuestas de chat, etc.) 
status, data = call("GET", "/notifications", token=tokens["ana"])
for n in data["data"]["content"]:
    leido = "✓" if n["is_read"] else "●"
    print(f"{leido} [{n['type']:<10}] {n['title']} - {n['message']}")

GET /notifications -> 200
● [ORDER     ] Pedido confirmado - Tu pedido ORD-1001 ha sido confirmado y está siendo procesado.
● [ORDER     ] Actualización de tu pedido - Tu pedido ORD-1001-1 ahora está en estado PREPARING.
● [ORDER     ] Actualización de tu pedido - Tu pedido ORD-1001-1 ahora está en estado SHIPPED.
● [ORDER     ] Actualización de tu pedido - Tu pedido ORD-1001-1 ahora está en estado DELIVERED.


In [45]:
#  Notificaciones del admin de TechStore (debe incluir 'Nuevo pedido recibido') 
status, data = call("GET", "/notifications", token=tokens["admin_techstore"])
for n in data["data"]["content"]:
    leido = "✓" if n["is_read"] else "●"
    print(f"{leido} [{n['type']:<10}] {n['title']} - {n['message']}")

GET /notifications -> 200
● [ORDER     ] Nuevo pedido recibido - Has recibido un nuevo pedido ORD-1001-1 en TechStore HN.


In [46]:
#  Contador de no leídas (para un badge en el frontend) 
status, data = call("GET", "/notifications/unread-count", token=tokens["admin_techstore"])
pretty(data)

GET /notifications/unread-count -> 200
{
  "success": true,
  "message": "OK",
  "data": {
    "unread_count": 1
  },
  "timestamp": "2026-06-18T23:04:16.835654Z"
}


In [47]:
#  Historial de auditoría (Super Admin): todas las acciones administrativas 
status, data = call("GET", "/admin/audit-logs", token=tokens["super_admin"])
for log in data["data"]["content"]:
    print(f"[{log['created_at']}] {log['action']:<28} | {log['entity_type']:<12} #{log['entity_id']}")

GET /admin/audit-logs -> 200
[2026-06-18T17:04:15] STORE_ORDER_STATUS_CHANGED   | STORE_ORDER  #1
[2026-06-18T17:04:15] STORE_ORDER_STATUS_CHANGED   | STORE_ORDER  #1
[2026-06-18T17:04:15] STORE_ORDER_STATUS_CHANGED   | STORE_ORDER  #1


## 10. Gestión administrativa (Super Admin)
capacidades exclusivas del Super Administrador: crear tiendas/admins, listar usuarios, suspender una tienda.

In [48]:
#  Listar todos los usuarios de la plataforma 
status, data = call("GET", "/admin/users", token=tokens["super_admin"])
for u in data["data"]["content"]:
    print(f"#{u['id']} | {u['full_name']:<20} | {u['email']:<30} | {u['role']['name']} | {u['status']}")

GET /admin/users -> 200
#1 | Carlos Rivera        | superadmin@arix.com            | ROLE_SUPER_ADMIN | ACTIVE
#2 | María Fernández      | admin.techstore@arix.com       | ROLE_STORE_ADMIN | ACTIVE
#3 | Jorge Martínez       | admin.modacasa@arix.com        | ROLE_STORE_ADMIN | ACTIVE
#4 | Ana López            | ana.lopez@gmail.com            | ROLE_CLIENT | ACTIVE
#5 | Luis Gómez           | luis.gomez@gmail.com           | ROLE_CLIENT | ACTIVE
#6 | Karla Suárez         | karla.suarez@gmail.com         | ROLE_CLIENT | ACTIVE
#7 | Pedro Martínez       | GustavoDiaz@gmail.com          | ROLE_CLIENT | ACTIVE
#8 | Hilery               | Hilery@arix.com                | ROLE_CLIENT | ACTIVE
#9 | Cliente Demo         | cliente.demo.1781823855@gmail.com | ROLE_CLIENT | ACTIVE


In [49]:
#  Listar todas las tiendas (incluye estado) 
status, data = call("GET", "/admin/stores", token=tokens["super_admin"])
for s in data["data"]["content"]:
    print(f"#{s['id']} | {s['business_name']:<20} | {s['status']}")

GET /admin/stores -> 200
#1 | TechStore HN         | ACTIVE
#2 | ModaCasa             | ACTIVE


In [50]:
#  Suspender temporalmente ModaCasa (demuestra moderación de tiendas) 
status, data = call("PUT", f"/admin/stores/{ids['modacasa_id']}/status", token=tokens["super_admin"], json={"status": "SUSPENDED"})
print(f"Nuevo estado de ModaCasa: {data['data']['status']}")

PUT /admin/stores/2/status -> 200
Nuevo estado de ModaCasa: SUSPENDED


In [51]:
#  Reactivar ModaCasa 
status, data = call("PUT", f"/admin/stores/{ids['modacasa_id']}/status", token=tokens["super_admin"], json={"status": "ACTIVE"})
print(f"Nuevo estado de ModaCasa: {data['data']['status']}")

PUT /admin/stores/2/status -> 200
Nuevo estado de ModaCasa: ACTIVE


## Resumen
los 13 módulos implementados:

- **users**: registro, login (3 roles), perfil
- **stores**: marketplace público, gestión de tienda propia
- **categories / products / inventory**: catálogo, filtros, stock
- **orders**: checkout multi-tienda con generación automática de subórdenes
- **invoices**: generación y descarga de PDF (por tienda y consolidada)
- **reviews / favorites**: reseñas restringidas a compradores, favoritos
- **tickets**: soporte con cambio de estado automático
- **chat**: mensajería persistente (+ arquitectura WebSocket documentada)
- **notifications / audit**: trazabilidad administrativa